# From ML Fundamentals to Deep Learning (scikit-learn)

How to use this course: read the Markdown explanations, **run the code yourself**, and explore how parameters affect the results. Then complete the 📝 exercises in the empty cells. Writing the code yourself helps reinforce what you learn.

**Contents:**
0. Data types and terminology
1. The basic workflow: fit / predict, train/test, overfitting
2. Regression and its metrics
3. Classification and its metrics (accuracy, precision, recall, F1, ROC AUC)
4. Cross-validation and data leakage
5. Preprocessing: scaling, encoding, missing values, Pipeline
6. Regularization, kNN, SVM
7. Trees, Random Forest, Boosting, feature importance
8. Hyperparameter tuning
9. Imbalanced classes
10. Unsupervised learning: KMeans, PCA, DBSCAN
11. A bridge to Deep Learning: gradient descent, MLP
12. Final project

Also included: a self-assessment checklist and next steps in Deep Learning.

Prerequisites: basic familiarity with pandas, scikit-learn, and decision trees. The course organizes fundamental concepts and gradually prepares you for Deep Learning.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
pd.set_option("display.max_columns", 50)
print("ok")

## 0. Data types and terminology

### Column types
Python types (`int`, `float`, `str`) are not the same as **ML feature types**. What matters to a model is what a column *represents*:

| ML type | Example | Pandas dtype | Model input preparation |
|---|---|---|---|
| **Numerical continuous** | price, weight, temperature | float64 | as is (+ scaling for some models) |
| **Numerical discrete** | number of rooms, age in years | int64 | as is |
| **Nominal categorical** (unordered) | city, color, brand | object / category | One-Hot Encoding |
| **Ordinal categorical** (ordered) | S < M < L, education level | object / category | OrdinalEncoder with an explicit order |
| **Binary** | yes/no, 0/1 | bool / int | as is |
| **Datetime** | order date | datetime64 | extract year/month/day of week, etc. |
| **Text** | review | object | TF-IDF / embeddings |

⚠️ A common pitfall: `city_id = 1,2,3` has an integer dtype but is **nominal**. A model should not assume that city 3 is "greater" than city 1.

### Terminology
- **X**: features; **y**: the target variable.
- **Supervised learning**: y is available. A numerical target → **regression**; a class label → **classification**.
- **Unsupervised learning**: no y; discover structure (clusters, dimensionality reduction).
- **Train / test**: fit on training data and evaluate on test data the model has never seen.
- **Parameters** are learned by the model (e.g., coefficients). **Hyperparameters** are set before training (e.g., tree depth, k in kNN).


In [ ]:
df = pd.DataFrame({
    "age": [25, 32, 47, 51],                       # int, discrete numeric
    "income": [1200.5, 3400.0, 2100.75, 5000.0],   # float, continuous
    "city": ["Lviv", "Kyiv", "Lviv", "Odesa"],     # nominal
    "size": ["S", "M", "L", "M"],                  # ordinal
    "is_vip": [True, False, False, True],          # binary
    "city_id": [1, 2, 1, 3],                       # int dtype, but actually nominal!
})
print(df.dtypes)
print()
print(df.select_dtypes(include="number").columns.tolist())
print(df.select_dtypes(include="object").columns.tolist())
df.describe(include="all")

### 📝 Exercise 0.1
Create a `size_num` column mapping S→0, M→1, L→2 (manual ordinal encoding using `.map`). Then run `pd.get_dummies(df['city'])` and inspect the result. Why should city not simply be encoded as Lviv→0, Kyiv→1, Odesa→2?


In [ ]:
# Your solution here

## 1. The basic workflow: fit / predict, train/test, overfitting

Scikit-learn is built around a consistent API:

```python
model = SomeModel(hyperparameters)
model.fit(X_train, y_train)      # train
y_pred = model.predict(X_test)  # predict
model.score(X_test, y_test)      # quick evaluation
```
Preprocessors provide `transform` (and `fit_transform`) instead of predict.

### Overfitting vs underfitting
- **Underfitting** (high bias): the model is too simple and performs poorly on both train and test.
- **Overfitting** (high variance): the model memorizes the training data, performing well on train but poorly on test.
- The goal is **strong performance on unseen data**, rather than training performance alone.

Explore this using a decision tree with different depths.


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print(X.shape, y.value_counts(normalize=True).round(2).to_dict())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

depths = range(1, 21)
tr, te = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    tr.append(m.score(X_train, y_train))
    te.append(m.score(X_test, y_test))

plt.plot(depths, tr, label="train")
plt.plot(depths, te, label="test")
plt.xlabel("max_depth"); plt.ylabel("accuracy"); plt.legend(); plt.grid(alpha=.3)
plt.title("Overfitting: train improves, test plateaus/drops")
plt.show()

`stratify=y` preserves class proportions in train and test. Use stratification for random classification splits. `random_state` makes the split reproducible.


### 📝 Exercise 1.1
Repeat the experiment, varying `min_samples_leaf` (1..30) with unrestricted tree depth. Plot the results. Which value gives the best test score? Is this a reliable choice, given a single small test set? Think about it; Section 4 revisits this question.


In [ ]:
# Your solution here

## 2. Regression and its metrics

Regression predicts a number. We will use `load_diabetes`.

### Regression metrics
Let `e_i = y_i − ŷ_i` be the prediction error.
- **MAE** = mean |e|. In target units, easy to interpret, and less sensitive to outliers than squared error.
- **MSE** = mean e². Strongly penalizes large errors.
- **RMSE** = √MSE. Also in target units, but sensitive to outliers.
- **R²** = 1 − SS_res / SS_tot. Measures the fraction of variance in y explained by the model. 1 is perfect; 0 matches always predicting the mean. It **can be negative** (worse than predicting the mean).

Always compare against a **baseline** (`DummyRegressor`): "R²=0.4" alone does not tell the whole story.


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Xd, yd = load_diabetes(return_X_y=True, as_frame=True)
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.2, random_state=0)

def report(name, model):
    model.fit(Xtr, ytr)
    p = model.predict(Xte)
    mse = mean_squared_error(yte, p)
    print(f"{name:12s} MAE={mean_absolute_error(yte,p):6.2f}  RMSE={np.sqrt(mse):6.2f}  R2={r2_score(yte,p):5.3f}")

report("baseline", DummyRegressor(strategy="mean"))
report("linear", LinearRegression())

lin = LinearRegression().fit(Xtr, ytr)
print(pd.Series(lin.coef_, index=Xd.columns).round(1).sort_values())
print("intercept:", round(lin.intercept_, 1))

**Interpreting linear regression:** `ŷ = b0 + b1·x1 + b2·x2 + ...`. A coefficient describes how much the prediction changes when its feature increases by 1, with the other features held fixed. Here the features have already been standardized, so their coefficients can be compared. In general, coefficient magnitudes are only comparable when feature scales are comparable.

Useful plots: **predicted vs actual** and **residuals**.


In [ ]:
p = lin.predict(Xte)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(yte, p, alpha=.6); lim = [yte.min(), yte.max()]
ax[0].plot(lim, lim, "r--"); ax[0].set_xlabel("actual"); ax[0].set_ylabel("predicted")
ax[1].scatter(p, yte - p, alpha=.6); ax[1].axhline(0, color="r", ls="--")
ax[1].set_xlabel("predicted"); ax[1].set_ylabel("residual")
plt.show()

### 📝 Exercise 2.1
Calculate MAE, MSE, RMSE, and R² **manually using NumPy** (without sklearn.metrics) and compare them with sklearn's results. Then add one outlier to `yte` (for example, add 1000 to one value) and see which metric reacts most strongly.


In [ ]:
# Your solution here

### 📝 Exercise 2.2
Compare `LinearRegression`, `DecisionTreeRegressor(max_depth=3)`, and `RandomForestRegressor(n_estimators=200)` using R² and MAE on this dataset. Which performs best? Is it substantially better than linear regression?


In [ ]:
# Your solution here

## 3. Classification and its metrics (accuracy, precision, recall, F1, ROC AUC)

### Confusion matrix (binary classification)
|  | Predicted 1 | Predicted 0 |
|---|---|---|
| **Actual 1** | TP | FN |
| **Actual 0** | FP | TN |

### Metrics
- **Accuracy** = (TP+TN)/total. The fraction of correct predictions. ❌ Misleading with imbalanced classes: if 99% of examples belong to class 0, always predicting 0 achieves 99% accuracy.
- **Precision** = TP/(TP+FP). The fraction of predicted positives that are actually positive. Relevant when **false alarms** are costly (e.g., marking an important email as spam).
- **Recall** (sensitivity, TPR) = TP/(TP+FN). The fraction of actual positives detected. Relevant when **missing a positive** is costly (e.g., disease or fraud).
- **F1** = the harmonic mean of precision and recall. Summarizes both in one value.
- **ROC AUC**: vary the probability threshold from 0 to 1 and plot TPR vs FPR. AUC is the probability that a randomly selected positive ranks above a randomly selected negative. 0.5 corresponds to random ranking; 1.0 is perfect. It does not depend on one chosen threshold.
- **PR AUC / Average Precision**: often more informative than ROC AUC under severe class imbalance.
- **Log loss**: penalizes confidently incorrect probabilities; logistic regression minimizes it during training.

### Decision threshold
For binary probability predictions, 0.5 is a common decision threshold, but **it is a convention**. Lowering the threshold generally increases recall at the cost of precision. The application determines the appropriate threshold.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score,
                             classification_report, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

print("accuracy :", round(float(accuracy_score(y_test, pred)), 3))
print("precision:", round(float(precision_score(y_test, pred)), 3))
print("recall   :", round(float(recall_score(y_test, pred)), 3))
print("f1       :", round(float(f1_score(y_test, pred)), 3))
print("roc auc  :", round(float(roc_auc_score(y_test, proba)), 3))
print()
print(classification_report(y_test, pred, target_names=data.target_names))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax[0], display_labels=data.target_names)
RocCurveDisplay.from_predictions(y_test, proba, ax=ax[1])
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=ax[2])
plt.show()

In [ ]:
# How the threshold changes precision/recall
for thr in [0.1, 0.3, 0.5, 0.7, 0.9]:
    p = (proba >= thr).astype(int)
    print(f"thr={thr}  precision={precision_score(y_test,p):.3f}  recall={recall_score(y_test,p):.3f}  f1={f1_score(y_test,p):.3f}")

### Multiclass classification and averaging
For 3+ classes, precision/recall/F1 are computed per class and then averaged:
- `macro`: unweighted mean across classes (all classes count equally)
- `micro`: aggregates TP/FP/FN globally (equivalent to accuracy for single-label classification across all classes)
- `weighted`: averages with weights proportional to class support


### 📝 Exercise 3.1
Use `load_iris` (3 classes), train logistic regression, and display a confusion matrix and `classification_report`. Calculate macro-F1 manually from per-class precision and recall.


In [ ]:
# Your solution here

### 📝 Exercise 3.2
For breast_cancer: in cancer diagnosis, which matters more, precision or recall? Find a threshold where recall ≥ 0.99 and inspect the corresponding precision. How many healthy people were incorrectly flagged?


In [ ]:
# Your solution here

### 📝 Exercise 3.3
Manually create `y_true = [0]*95 + [1]*5` and `y_pred = [0]*100`. Calculate accuracy, precision, recall, and f1. Explain in your own words why accuracy can be misleading here.


In [ ]:
# Your solution here

## 4. Cross-validation and data leakage

A single train/test split gives a **noisy** estimate: the result depends on the particular split. **k-fold cross-validation** divides the data into k parts, trains on k−1 parts and evaluates on the remaining part in turn, then averages the results.

- `KFold`: standard folds
- `StratifiedKFold`: preserves class proportions (the usual default for classification)
- `GroupKFold`: keeps groups together (e.g., multiple records from one patient must not appear in both train and test)
- `TimeSeriesSplit`: for time series (no access to future information)

### Data leakage: a cause of unreliable evaluation
Leakage occurs when information from the test set, the future, or the target enters training inappropriately. Evaluation scores become inflated, and practical performance deteriorates.

Common examples:
1. **Fitting a scaler/imputer/encoder on all data** before splitting. Fit only on train. Use a **Pipeline**.
2. A feature that results from the target (e.g., `refund_issued` for predicting "customer is dissatisfied").
3. Duplicates shared between train and test.
4. Tuning hyperparameters on test (which invalidates it as an independent evaluation). Use a separate validation set or CV.


In [ ]:
from sklearn.model_selection import cross_val_score, cross_validate, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=cv, scoring="roc_auc")
print("AUC by fold:", scores.round(3))
print(f"mean={scores.mean():.3f}  std={scores.std():.3f}")

res = cross_validate(clf, X, y, cv=cv, scoring=["accuracy", "precision", "recall", "f1"], return_train_score=True)
pd.DataFrame(res).mean().round(3)

In [ ]:
# Leakage demonstration: scaling all data vs using a Pipeline.
# The StandardScaler effect is small, but the principle is the same. Stronger leakage: target-based feature selection BEFORE CV.
from sklearn.feature_selection import SelectKBest, f_classif

rng = np.random.RandomState(0)
X_noise = rng.randn(100, 2000)          # pure noise
y_noise = rng.randint(0, 2, 100)        # random labels: expected AUC without leakage is 0.5

# ❌ incorrect: select the 20 "best" features on ALL data, then run CV
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
bad = cross_val_score(LogisticRegression(), X_sel, y_noise, cv=5, scoring="roc_auc").mean()

# ✅ correct: selection inside Pipeline (fit only on the training fold)
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
good = cross_val_score(pipe, X_noise, y_noise, cv=5, scoring="roc_auc").mean()

print(f"with leakage  AUC = {bad:.2f}   (on pure noise!)")
print(f"no leakage    AUC = {good:.2f}   (approximately 0.5, as expected)")

### 📝 Exercise 4.1
Compare 3 models (LogReg in a pipeline with a scaler, DecisionTree(max_depth=5), and RandomForest) on breast_cancer using 5-fold CV and F1. Report mean ± std. Are the differences between models larger than the standard deviations?


In [ ]:
# Your solution here

### 📝 Exercise 4.2
Return to Exercise 1.1 and tune `min_samples_leaf` using CV instead of a single test set. Does the selected value change?


In [ ]:
# Your solution here

## 5. Preprocessing: scaling, encoding, missing values, Pipeline

### Scaling
- Important for models that rely on distances or gradient optimization: **kNN, SVM, regularized logistic regression, neural networks, PCA, KMeans**.
- **Not required** for trees, forests, or tree boosting (their splits depend on value ordering).
- `StandardScaler`: (x − mean)/std. `MinMaxScaler`: maps to [0,1]. `RobustScaler`: uses the median and IQR to reduce sensitivity to outliers.

### Categorical encoding
- `OneHotEncoder(handle_unknown="ignore")`: for nominal features. Each category becomes a 0/1 column.
- `OrdinalEncoder(categories=[[...]])`: for ordinal features with an explicit order.
- With hundreds of categories, One-Hot can greatly expand the matrix. Alternatives include target encoding, hashing, or tree models with native categorical support (HistGradientBoosting).

### Missing values
- `SimpleImputer(strategy="median" / "most_frequent")`.
- `add_indicator=True` adds a missingness indicator, because the absence of a value may itself carry information.

### ColumnTransformer + Pipeline
Different columns need different processing. `ColumnTransformer` applies each transformer to the appropriate columns. `Pipeline` combines preprocessing and the model into **one object** fitted only on the training portion of each CV fold. This is a standard industry workflow.


In [ ]:
# Synthetic real-world-style data with mixed types and missing values
rng = np.random.RandomState(1)
n = 1500
d = pd.DataFrame({
    "age": rng.randint(18, 70, n).astype(float),
    "income": rng.lognormal(8, 0.6, n),
    "city": rng.choice(["Lviv", "Kyiv", "Odesa", "Kharkiv"], n),
    "plan": rng.choice(["basic", "plus", "pro"], n, p=[.5, .3, .2]),
    "tickets": rng.poisson(2, n),
})
logit = 0.03*(d.age-40) + 0.0003*(d.income-3000) + 0.4*d.tickets \
        + d.plan.map({"basic": 0.8, "plus": 0, "pro": -0.8}) + (d.city == "Kyiv")*0.5 - 1.5
d["churn"] = (rng.rand(n) < 1/(1+np.exp(-logit))).astype(int)
d.loc[rng.rand(n) < .1, "age"] = np.nan
d.loc[rng.rand(n) < .08, "city"] = np.nan
print(d.isna().mean().round(3).to_dict(), "| churn rate:", d.churn.mean().round(2))
d.head()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

Xc = d.drop(columns="churn"); yc = d["churn"]
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=42, stratify=yc)

num_cols = ["age", "income", "tickets"]
nom_cols = ["city"]
ord_cols = ["plan"]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                      ("sc", StandardScaler())]), num_cols),
    ("nom", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), nom_cols),
    ("ord", OrdinalEncoder(categories=[["basic", "plus", "pro"]]), ord_cols),
])

model = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
model.fit(Xc_tr, yc_tr)
print("LogReg AUC:", round(float(roc_auc_score(yc_te, model.predict_proba(Xc_te)[:, 1])), 3))

# inspect the preprocessing output
print(model.named_steps["pre"].get_feature_names_out())

### 📝 Exercise 5.1
Replace `LogisticRegression` with `RandomForestClassifier` in the same pipeline. Compare AUC using 5-fold CV. Does the forest need StandardScaler here? Remove it and check whether the result changes.


In [ ]:
# Your solution here

### 📝 Exercise 5.2
Add a `signup_date` column to the dataset (random dates over 2 years), extract `month` and `dayofweek` (using `FunctionTransformer` or pandas before the pipeline), and include them in the model.


In [ ]:
# Your solution here

## 6. Regularization, kNN, SVM

### Regularization (reducing overfitting in linear models)
Add a penalty for large coefficients to the loss:
- **Ridge (L2)**: penalizes Σw². Shrinks coefficients without forcing them to zero. Useful with multicollinearity.
- **Lasso (L1)**: penalizes Σ|w|. **Sets some coefficients to zero**, performing feature selection.
- **ElasticNet**: combines both penalties.

Regularization strength: `alpha` in Ridge/Lasso (larger = stronger), but **`C` in `LogisticRegression` is inverse strength** (smaller C = stronger regularization). This distinction is easy to miss.

### kNN
Finds the k nearest training examples for a new point and votes (classification) or averages (regression). Requires scaling. Small k can overfit; large k can underfit. Can be slow on large datasets.

### SVM
Finds a separating hyperplane with the largest margin. `kernel="rbf"` enables nonlinear boundaries. Main hyperparameters: `C` and `gamma`. Requires scaling.


In [ ]:
from sklearn.linear_model import Ridge, Lasso
Xs = StandardScaler().fit_transform(Xd)   # (for demonstration; use a Pipeline in real applications!)
alphas = np.logspace(-3, 1, 30)
ridge_coefs, lasso_coefs = [], []
for a in alphas:
    ridge_coefs.append(Ridge(alpha=a).fit(Xs, yd).coef_)
    lasso_coefs.append(Lasso(alpha=a).fit(Xs, yd).coef_)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(alphas, ridge_coefs); ax[0].set_xscale("log"); ax[0].set_title("Ridge: coefficients shrink")
ax[1].plot(alphas, lasso_coefs); ax[1].set_xscale("log"); ax[1].set_title("Lasso: coefficients become 0")
for a in ax: a.set_xlabel("alpha")
plt.show()

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

for name, m in {
    "kNN k=5": make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
    "kNN k=5 WITHOUT scaler": KNeighborsClassifier(5),
    "SVM rbf": make_pipeline(StandardScaler(), SVC()),
    "SVM rbf WITHOUT scaler": SVC(),
}.items():
    s = cross_val_score(m, X, y, cv=cv, scoring="accuracy").mean()
    print(f"{name:22s} acc={s:.3f}")

### 📝 Exercise 6.1
Build a validation curve on breast_cancer: vary k=1..40 for kNN and plot CV accuracy. Where does overfitting or underfitting occur? See `sklearn.model_selection.validation_curve`.


In [ ]:
# Your solution here

### 📝 Exercise 6.2
Train `LogisticRegression(penalty='l1', solver='liblinear')` with different C values on breast_cancer. How many coefficients become exactly 0 for each C?


In [ ]:
# Your solution here

## 7. Trees, Random Forest, Boosting, feature importance

### Decision trees
Recursively split the feature space using a threshold on one feature to reduce **impurity** (Gini or entropy for classification, MSE for regression). A single deep tree tends to overfit (high variance).
Hyperparameters: `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`, `ccp_alpha` (pruning).

### Ensembles: two distinct approaches
- **Bagging → Random Forest**: many trees, each trained on a bootstrap sample and using random feature subsets; predictions are averaged. **Reduces variance**. Trees are independent and can be trained in parallel. Often resistant to overfitting.
- **Boosting → Gradient Boosting (GradientBoosting, HistGradientBoosting, XGBoost, LightGBM, CatBoost)**: trees are built **sequentially**, each correcting earlier errors by following the loss gradient. **Reduces bias**. More sensitive to hyperparameters (`learning_rate`, `n_estimators`, `max_depth`), and often very effective on tabular data.

### Feature importance
- `feature_importances_` (impurity-based): fast, but **biased** toward features with many unique values.
- **Permutation importance**: shuffle one feature on test and measure the performance drop. Provides a more direct assessment and works with any model.


In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.tree import plot_tree

models = {
    "tree (depth=None)": DecisionTreeClassifier(random_state=0),
    "tree (depth=4)": DecisionTreeClassifier(max_depth=4, random_state=0),
    "random forest": RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
    "hist gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
for n_, m in models.items():
    s = cross_val_score(m, X, y, cv=cv, scoring="roc_auc")
    print(f"{n_:24s} AUC={s.mean():.3f} ± {s.std():.3f}")

plt.figure(figsize=(14, 6))
t = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
plot_tree(t, feature_names=X.columns, class_names=data.target_names, filled=True, fontsize=8)
plt.show()

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(X_train, y_train)
imp_gini = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=0, scoring="roc_auc")
imp_perm = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
imp_gini.head(10)[::-1].plot.barh(ax=ax[0], title="impurity importance")
imp_perm.head(10)[::-1].plot.barh(ax=ax[1], title="permutation importance (test)")
plt.tight_layout(); plt.show()

### 📝 Exercise 7.1
Explore Random Forest: how does CV AUC change for `n_estimators` = 1, 5, 20, 100, 500? Where does it plateau? What happens with `max_features=1` vs `max_features='sqrt'` vs `max_features=None`?


In [ ]:
# Your solution here

### 📝 Exercise 7.2
Boosting: train `GradientBoostingClassifier` with `learning_rate` ∈ {0.5, 0.1, 0.01} and `n_estimators`=300. Plot train/test error over iterations using `staged_predict_proba` or `staged_predict`. What relationship do you observe between learning_rate and the number of trees?


In [ ]:
# Your solution here

## 8. Hyperparameter tuning

- `GridSearchCV`: tries every combination in a grid. Suitable for 1–3 parameters.
- `RandomizedSearchCV`: samples combinations from distributions. More efficient with many parameters.
- In a Pipeline, parameters use the form `step__parameter`.
- Tune on **train using CV**, and evaluate on test **once at the end**.

The final evaluation of the selected model on untouched test data provides an independent estimate. CV's `best_score_` is somewhat optimistic because it is the maximum selected from many trials.


In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import randint, uniform

pipe = Pipeline([("sc", StandardScaler()), ("clf", SVC(probability=True))])
grid = {
    "clf__C": [0.1, 1, 10, 100],
    "clf__gamma": ["scale", 0.01, 0.001],
}
gs = GridSearchCV(pipe, grid, cv=5, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
print("best params:", gs.best_params_)
print("cv auc     :", gs.best_score_.round(4))
print("test auc   :", round(float(roc_auc_score(y_test, gs.predict_proba(X_test)[:, 1])), 4))

pd.DataFrame(gs.cv_results_)[["params", "mean_test_score", "std_test_score", "rank_test_score"]].sort_values("rank_test_score").head()

In [ ]:
rs = RandomizedSearchCV(
    RandomForestClassifier(random_state=0, n_jobs=-1),
    {"n_estimators": randint(100, 600), "max_depth": randint(3, 20),
     "min_samples_leaf": randint(1, 10), "max_features": uniform(0.1, 0.9)},
    n_iter=25, cv=5, scoring="roc_auc", random_state=0, n_jobs=-1,
).fit(X_train, y_train)
print(rs.best_params_)
print("cv:", rs.best_score_.round(4), "| test:", round(float(roc_auc_score(y_test, rs.predict_proba(X_test)[:, 1])), 4))

### 📝 Exercise 8.1
Tune `HistGradientBoostingClassifier` (learning_rate, max_depth, max_leaf_nodes, l2_regularization) with RandomizedSearchCV. Compare test AUC with the default model. Is the improvement worth the time?


In [ ]:
# Your solution here

### 📝 Exercise 8.2
Theory: why should hyperparameters not be tuned on test? Give an example showing how this inflates performance estimates. Hint: 1000 random models on a small test set.


In [ ]:
# Your solution here

## 9. Imbalanced classes

When positives make up 1–5% of examples (fraud, churn, rare disease):
1. **Do not rely on accuracy alone.** Use PR AUC (`average_precision`), F1, or recall at a specified precision.
2. Use `stratify` in the split and `StratifiedKFold` in CV.
3. **`class_weight="balanced"`** increases the contribution of the rare class to the loss (a simple and often useful starting point).
4. **Threshold**: tune it for the business metric.
5. Oversampling/undersampling (`imbalanced-learn`, SMOTE) can help, but apply it **only to train, inside the CV pipeline** to avoid leakage.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.metrics import average_precision_score

Xi, yi = make_classification(n_samples=20000, n_features=20, n_informative=6,
                             weights=[0.98, 0.02], flip_y=0.01, random_state=0)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3, stratify=yi, random_state=0)
print("positive fraction:", yi.mean().round(3))

for cw in [None, "balanced"]:
    m = make_pipeline(StandardScaler(), LogisticRegression(class_weight=cw, max_iter=1000)).fit(Xi_tr, yi_tr)
    pr = m.predict(Xi_te); pb = m.predict_proba(Xi_te)[:, 1]
    print(f"class_weight={str(cw):9s} acc={accuracy_score(yi_te,pr):.3f} prec={precision_score(yi_te,pr):.3f} "
          f"rec={recall_score(yi_te,pr):.3f} f1={f1_score(yi_te,pr):.3f} "
          f"rocAUC={roc_auc_score(yi_te,pb):.3f} PR-AUC={average_precision_score(yi_te,pb):.3f}")

### 📝 Exercise 9.1
Compare RandomForest with `class_weight=None`, `'balanced'`, and `'balanced_subsample'` on this dataset. Plot the PR curves together.


In [ ]:
# Your solution here

### 📝 Exercise 9.2
For the best model, select a threshold that maximizes F1 (use `precision_recall_curve`). What is the threshold, and why is it not 0.5?


In [ ]:
# Your solution here

## 10. Unsupervised learning: KMeans, PCA, DBSCAN

### KMeans
Choose k. The algorithm alternates between assigning points to the nearest centroid and updating centroids. It assumes roughly spherical clusters of similar size. Scaling is needed. Choose k using the **elbow** method (inertia) and **silhouette score** (−1 to 1; higher is better).

### DBSCAN
Clusters are dense regions. The algorithm determines the number of clusters, identifies **noise (-1)**, and can detect arbitrary shapes. Parameters: `eps`, `min_samples`.

### PCA
Finds new axes (principal components) in directions of maximum variance. Uses include compression, 2D visualization, and reducing noise or multicollinearity. Scaling is needed. `explained_variance_ratio_` reports how much variance each component explains.


In [ ]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs, make_moons

Xb, _ = make_blobs(n_samples=600, centers=4, cluster_std=1.0, random_state=3)
ks = range(2, 10)
inertia = [KMeans(k, n_init=10, random_state=0).fit(Xb).inertia_ for k in ks]
sil = [silhouette_score(Xb, KMeans(k, n_init=10, random_state=0).fit_predict(Xb)) for k in ks]

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(ks, inertia, "o-"); ax[0].set_title("elbow (inertia)")
ax[1].plot(ks, sil, "o-"); ax[1].set_title("silhouette")
ax[2].scatter(*Xb.T, c=KMeans(4, n_init=10, random_state=0).fit_predict(Xb), s=10); ax[2].set_title("KMeans k=4")
plt.show()

# KMeans vs DBSCAN on the moons dataset
Xm, _ = make_moons(400, noise=.06, random_state=0)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(*Xm.T, c=KMeans(2, n_init=10, random_state=0).fit_predict(Xm), s=10); ax[0].set_title("KMeans: unsuitable cluster shapes")
ax[1].scatter(*Xm.T, c=DBSCAN(eps=.2, min_samples=5).fit_predict(Xm), s=10); ax[1].set_title("DBSCAN: captures the cluster shapes")
plt.show()

In [ ]:
pca = PCA().fit(StandardScaler().fit_transform(X))
cum = np.cumsum(pca.explained_variance_ratio_)
print("components needed for 95% variance:", int(np.argmax(cum >= .95)) + 1, "out of", X.shape[1])

Z = PCA(2).fit_transform(StandardScaler().fit_transform(X))
plt.scatter(Z[:, 0], Z[:, 1], c=y, cmap="coolwarm", s=12, alpha=.7)
plt.title("breast_cancer in 2D (PCA)"); plt.show()

### 📝 Exercise 10.1
Use `load_wine`, scale the features, run KMeans with k=3, and compare clusters with actual labels using `adjusted_rand_score`. What happens without scaling?


In [ ]:
# Your solution here

### 📝 Exercise 10.2
Build a `StandardScaler → PCA(n_components=0.95) → LogisticRegression` pipeline on breast_cancer and compare CV AUC with logistic regression without PCA.


In [ ]:
# Your solution here

## 11. A bridge to Deep Learning: gradient descent, MLP

A neural network builds on **linear models arranged in layers with nonlinearities** and is trained using **gradient descent**. We will explore the process manually.

### Key ideas
1. **Loss function**: a number measuring prediction error (MSE, cross-entropy).
2. **Gradient descent**: compute the loss gradient with respect to the parameters and step in the opposite direction: `w ← w − lr · ∂L/∂w`.
3. **Learning rate**: step size. Too small → slow learning; too large → divergence.
4. **Epoch / batch**: an epoch is one pass through the data; mini-batches allow weight updates using smaller portions of data.
5. **Neuron**: `a = f(w·x + b)`, where f is a nonlinear activation (ReLU, sigmoid, tanh). Without nonlinearities, a sequence of linear layers collapses to one linear transformation.
6. **Backpropagation**: efficient gradient computation using the chain rule in a backward pass through the layers. PyTorch does this automatically (`loss.backward()`).
7. Everything covered so far (overfitting, regularization, CV, metrics, scaling) **also applies to DL**. Deep Learning extends these ideas.

Start with logistic regression implemented in NumPy: a single-neuron network.


In [ ]:
# Logistic regression from scratch: gradient descent
Xg = StandardScaler().fit_transform(X_train.values)
yg = y_train.values.astype(float)

def sigmoid(z): return 1 / (1 + np.exp(-z))

w = np.zeros(Xg.shape[1]); b = 0.0
lr, epochs = 0.1, 200
losses = []
for ep in range(epochs):
    p = sigmoid(Xg @ w + b)
    loss = -np.mean(yg*np.log(p + 1e-12) + (1-yg)*np.log(1-p + 1e-12))   # cross-entropy
    grad_w = Xg.T @ (p - yg) / len(yg)
    grad_b = np.mean(p - yg)
    w -= lr * grad_w; b -= lr * grad_b
    losses.append(loss)

plt.plot(losses); plt.xlabel("epoch"); plt.ylabel("log loss"); plt.title("Loss decreases"); plt.show()

Xg_te = StandardScaler().fit(X_train.values).transform(X_test.values)
acc = ((sigmoid(Xg_te @ w + b) > .5).astype(int) == y_test.values).mean()
print("custom logistic regression, test acc:", acc.round(3))

Next, a small neural network in scikit-learn: `MLPClassifier`, a fully connected network. Later, we will use **PyTorch** to study Deep Learning in more detail.


In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", alpha=1e-3,
                  early_stopping=True, max_iter=500, random_state=0),
).fit(X_train, y_train)
print("MLP test AUC:", round(float(roc_auc_score(y_test, mlp.predict_proba(X_test)[:, 1])), 4))

# loss curve
lossc = mlp.named_steps["mlpclassifier"].loss_curve_
plt.plot(lossc); plt.xlabel("iter"); plt.title("MLP training loss"); plt.show()

In [ ]:
# Why nonlinearities matter: XOR-like data
Xx = np.random.RandomState(0).randn(600, 2)
yx = ((Xx[:, 0] > 0) ^ (Xx[:, 1] > 0)).astype(int)
for name, m in {
    "logreg": LogisticRegression(),
    "MLP relu": MLPClassifier((16, 16), max_iter=2000, random_state=0),
}.items():
    print(f"{name:10s} acc =", cross_val_score(m, Xx, yx, cv=5).mean().round(3))

### 📝 Exercise 11.1
In the custom gradient descent implementation, set `lr` to 0.001, 0.1, 1, and 5. Plot all 4 loss curves together. What happens when lr is too large?


In [ ]:
# Your solution here

### 📝 Exercise 11.2
Extend the custom implementation with L2 regularization (`+ lam * w` in the gradient). How does the norm of `w` change for lam=0, 0.1, 1?


In [ ]:
# Your solution here

### 📝 Exercise 11.3
Implement a **mini-batch** version (batch_size=32, shuffle each epoch). Compare convergence speed with the full-batch version.


In [ ]:
# Your solution here

### 📝 Exercise 11.4
(Advanced) Implement a NumPy neural network with one hidden ReLU layer and a sigmoid output. Implement backpropagation manually and train it on the XOR data above.


In [ ]:
# Your solution here

## 12. Final project

Choose an open tabular dataset (e.g., Kaggle: Titanic, House Prices, Telco Churn, Credit Card Fraud) or educational online-store data for predicting customer purchases.

**Requirements (complete workflow):**
1. EDA: column types, missing values, distributions, target balance.
2. Clearly define the **metric** and **baseline** (Dummy).
3. Train/test split (stratified or time-based where appropriate). Keep test untouched.
4. `ColumnTransformer` + `Pipeline` for all preprocessing.
5. Compare 3+ models (LogReg, RandomForest, HistGradientBoosting) using CV.
6. Tune the best model with RandomizedSearchCV.
7. Select a threshold for the task, then evaluate on test **once**.
8. Permutation importance and a written interpretation: which features matter and why.
9. Save the model: `joblib.dump(pipeline, "model.joblib")`.

```python
# project skeleton
final = Pipeline([("pre", pre), ("clf", best_model)])
final.fit(X_train, y_train)
```


In [ ]:
# Your final project here

## ✅ Self-assessment checklist

To assess readiness for Deep Learning, try explaining these concepts independently:

- [ ] Nominal, ordinal, discrete, and continuous features and how to encode each
- [ ] Why accuracy can mislead with imbalance; precision/recall/F1 and ROC AUC
- [ ] Confusion matrices and how shifting the threshold changes precision/recall
- [ ] Why R² can be negative; MAE vs RMSE
- [ ] Overfitting/underfitting, bias/variance, and diagnosis using learning curves
- [ ] Data leakage and how Pipeline helps prevent it
- [ ] Why CV matters; when to use Stratified, Group, or TimeSeries splits
- [ ] Which models need scaling
- [ ] Ridge vs Lasso; why LogReg's `C` works inversely to `alpha`
- [ ] Bagging vs Boosting: their differences and effects on bias/variance
- [ ] Limitations of impurity importance compared with permutation importance
- [ ] Gradient descent: loss, gradient, lr, epoch, batch
- [ ] Why neural networks need nonlinear activations

## Next steps (Deep ML)
1. **PyTorch**: tensors, autograd, `nn.Module`, training loops (essentially Section 11 implemented in PyTorch).
2. Optimizers: SGD+momentum, Adam; lr scheduling; batch norm; dropout; weight decay.
3. CNNs (images), RNN/LSTM → Transformers (text and sequences).
4. Transfer learning and fine-tuning (Hugging Face).
5. In parallel: **XGBoost/LightGBM/CatBoost**, which often outperform neural networks on tabular data.

Books/courses: *Hands-On ML with Scikit-Learn, Keras & PyTorch* (Géron), Andrew Ng's ML Specialization, fast.ai, and Andrej Karpathy's "Neural Networks: Zero to Hero".
